In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
DATA_PATH = "../data/poker_hands.csv"            # <- adjust to where your file lives
CLASSES = ["nothing", "one_pair", "two_pairs", "three_of_a_kind", "straight",
           "flush", "full_house", "four_of_a_kind", "straight_flush", "royal_flush"]
SUITS = ["hearts", "diamonds", "clubs", "spades"]
RANK_ENCODING = "onehot"

In [2]:
df = pd.read_csv(DATA_PATH)
df = df.drop_duplicates().reset_index(drop=True)      # identical hands could otherwise end up in train AND test

def one_hot(cols, categories):
    return np.hstack([pd.get_dummies(pd.Categorical(df[c], categories=categories)).to_numpy(np.float32)
                      for c in cols])

suit_cols = [f"suit{i}" for i in range(1, 6)]
rank_cols = [f"rank{i}" for i in range(1, 6)]
suits = one_hot(suit_cols, SUITS)                                      # suits are categories, not numbers

if RANK_ENCODING == "onehot":
    ranks = one_hot(rank_cols, list(range(1, 14)))                     # lets the network "see" equal ranks
else:
    ranks = df[rank_cols].to_numpy(np.float32) / 13.0                  # ranks as one number scaled to 0..1

X = np.hstack([suits, ranks])
y = df["ranking"].map({c: i for i, c in enumerate(CLASSES)}).to_numpy()   # integer labels 0..9

print(X.shape, "features per hand")

(1022771, 85) features per hand


In [3]:
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

def balance(X, y, cap=20_000, floor=2_000, seed=SEED):
    """Undersample classes above `cap`; oversample (with replacement) classes below `floor`."""
    rng = np.random.default_rng(seed)
    idx = []
    for c in np.unique(y):
        ci = np.where(y == c)[0]
        if len(ci) > cap:
            ci = rng.choice(ci, cap, replace=False)
        elif len(ci) < floor:
            ci = rng.choice(ci, floor, replace=True)
        idx.append(ci)
    idx = rng.permutation(np.concatenate(idx))
    return X[idx], y[idx]

X_train, y_train = balance(X_tr, y_tr)
counts = lambda a: np.bincount(a, minlength=10)
print(pd.DataFrame({"class": CLASSES, "train (before)": counts(y_tr), "train (balanced)": counts(y_train),
                    "validation": counts(y_val), "test": counts(y_test)}).to_string(index=False))


          class  train (before)  train (balanced)  validation  test
        nothing          358803             20000       76887 76886
       one_pair          302511             20000       64824 64824
      two_pairs           34102             20000        7308  7307
three_of_a_kind           15109             15109        3238  3238
       straight            2778              2778         595   596
          flush            1432              2000         307   307
     full_house            1021              2000         218   219
 four_of_a_kind             165              2000          35    36
 straight_flush              12              2000           3     2
    royal_flush               6              2000           1     1


In [4]:
from tensorflow.keras.utils import to_categorical
Y_train, Y_val, Y_test = (to_categorical(a, 10) for a in (y_train, y_val, y_test))
np.savez_compressed("poker_prepared.npz", X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val,
                    X_test=X_test, y_test=y_test)